# Basics: Data Loading & GPS Visualization

This notebook demonstrates the fundamentals of loading and visualizing AIM™ telemetry data using libxrk.

## What You'll Find Here

- **Data Loading**: Load `.xrk` or `.xrz` files without AIM software
- **Lap Times Table**: View all recorded laps with computed lap times
- **GPS Speed Map**: Visualize speed around the track on an interactive map
- **Brake & Throttle Overlay**: See driver inputs overlaid on the track map

## Using Your Own Data

To analyze your own data:

1. **Run the first cell** below to install packages and display the upload widget
2. **Click "Choose File"** to select your `.xrk` or `.xrz` file
3. **Run all remaining cells** to analyze your data

The status indicator will show which file is being used. If you don't upload a file, the sample data will be used.

## Requirements

- GPS data channels (`GPS Latitude`, `GPS Longitude`, `GPS Speed`)
- Brake pressure (`BrakePress`) and throttle (`PPS`) for the inputs overlay

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [ ]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2 ipywidgets

# Import helper functions
from motorsports_data_notebook.visualization import (
    format_lap_time,
    get_best_lap_channels,
    plot_gps_channels,
    show_fig,
)
from motorsports_data_notebook.widgets import FileUpload, load_session

# File picker - upload your own .xrk/.xrz file or use the sample data
file_upload = FileUpload(default_file="CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz")
file_upload.display()

In [ ]:
# Load the data file with derived columns (speed_kmh, distance_m, lap_time)
log = load_session(file_upload.get_file_data())

# Get laps as pandas DataFrame for display
laps = log.laps.to_pandas()

In [ ]:
# Display lap times table
laps.style.format({"lap_time": format_lap_time})

In [ ]:
# Extract best lap channel data directly (faster than get_channels_as_table)
# Each channel keeps its native sample rate - no expensive merge/interpolation
best_lap, channels = get_best_lap_channels(
    log, laps, ["GPS Latitude", "GPS Longitude", "speed_kmh", "BrakePress", "PPS"]
)

In [ ]:
# Plot speed on GPS map using channel tables directly
fig = plot_gps_channels(
    channels,
    lat_channel="GPS Latitude",
    lon_channel="GPS Longitude",
    color_channels=[("speed_kmh", "Speed (km/h)", "Viridis")],
    title="Speed",
)
show_fig(fig)

In [ ]:
# Plot with multiple color channels (automatically interpolated to GPS timebase)
fig = plot_gps_channels(
    channels,
    lat_channel="GPS Latitude",
    lon_channel="GPS Longitude",
    color_channels=[
        ("BrakePress", "BrakePres", "Reds"),
        ("PPS", "Throttle", "Greens"),
    ],
    title="Accelerator and Brake Pressure on Best Lap",
)
show_fig(fig)